In [3]:
import pandas as pd
import re

# ── Paste your CSV path here ──────────────────────────────────────────────────
MRI_CSV = '../data/mri_reports.csv'
# ─────────────────────────────────────────────────────────────────────────────

STRUCTURAL_TYPES = [
    'MRI BRAIN',
    'MRI BRAIN WITH AND WITHOUT CONTRAST',
    'MRI BRAIN WITHOUT CONTRAST',
]

SECTION_SPLIT_PATTERN = re.compile(
    r'\n(?=(?:BRAIN MRI|HEAD MRA|NECK MRA|MRI BRAIN AND NECK'
    r'|SPINE|CERVICAL|THORACIC|LUMBAR|STRUCTURAL MRI|FUNCTIONAL MRI)\s*[:\n])',
    re.IGNORECASE
)
BRAIN_SECTION_PATTERN = re.compile(r'^(?:BRAIN MRI|MRI BRAIN|STRUCTURAL MRI)', re.IGNORECASE)
INLINE_HEADER_PATTERN = re.compile(r'^(?:BRAIN MRI|HEAD MRA|MRI BRAIN)\s*:?\s*\n', re.IGNORECASE)

BOILERPLATE_PATTERN = re.compile(
    r'forwarded to an automated communication'
    r'|findings were (?:discussed|communicated)'
    r'|carotid stenosis reference'
    r'|i,? the (?:attending|teaching) physician'
    r'|alert notification of critical'
    r'|critical results were communicated'
    r'|this report has been'
    r'|electronically notify'
    r'|study (?:terminated|aborted|discontinued) (?:early|due to|at patient)',
    re.IGNORECASE
)
HEADER_PATTERN = re.compile(
    r'^(?:brain mri|head mra|neck mra|mri brain|intracranial mra'
    r'|anterior circulation|posterior circulation|collateral circulation'
    r'|aortic\s+\[redacted\].*origin of major cervical'
    r'|impression|technique|comparison|indication|methodology'
    r'|detail|findings|neck):?\s*$',
    re.IGNORECASE
)
NEGATIVE_PATTERN = re.compile(
    r'^(?:there is |there are )?(?:no evidence of|no acute|no new|no abnormal'
    r'|no significant|unremarkable|within normal limits|no hemorrhage'
    r'|no infarct|no mass effect|no hydrocephalus|no midline shift'
    r'|patent|intact|clear|stable and unremarkable)'
    r'|^the (?:major |main )?(?:intracranial )?(?:flow voids|arterial flow voids|vessels).{0,60}(?:intact|preserved|patent)'
    r'|^(?:extracranial structures|paranasal sinuses|mastoid|orbits).{0,80}(?:clear|normal|unremarkable)',
    re.IGNORECASE
)
EXTRACRANIAL_PATTERN = re.compile(
    r'^(?:paranasal sinuses|mastoid air cells|orbits|skull base'
    r'|extracranial soft tissues|cervical|vertebral arter'
    r'|aortic|subclavian|carotid)',
    re.IGNORECASE
)

def is_noise(text):
    if len(text.strip()) < 30: return True
    if HEADER_PATTERN.match(text): return True
    if BOILERPLATE_PATTERN.search(text): return True
    return False

def is_negative(text): return bool(NEGATIVE_PATTERN.match(text))
def is_extracranial(text): return bool(EXTRACRANIAL_PATTERN.match(text))

def _extract_brain_section(text):
    if not SECTION_SPLIT_PATTERN.search(text):
        return text
    sections = SECTION_SPLIT_PATTERN.split(text)
    brain_sections = [s for s in sections if BRAIN_SECTION_PATTERN.match(s.strip())]
    return brain_sections[0] if brain_sections else sections[0]

def split_report(text):
    if pd.isna(text): return []
    text = _extract_brain_section(text)
    paragraphs = re.split(r'\n[ \t]*\n', text)
    result = []
    for p in paragraphs:
        p = p.strip()
        p = INLINE_HEADER_PATTERN.sub('', p).strip()
        if len(p) >= 30:
            result.append(p)
    return result

def split_report_old(text):
    if pd.isna(text): return []
    return [p.strip() for p in re.split(r'\n\s*\n', text) if p.strip()]

# ── Load ──────────────────────────────────────────────────────────────────────
print('Loading data...')
mri = pd.read_csv(MRI_CSV, engine='python', on_bad_lines='skip')
valid = mri[mri['findings'].notna() & ~mri['findings'].str.strip().str.lower().isin(['not available','none',''])]
structural = valid[valid['Type'].isin(STRUCTURAL_TYPES)].copy()
print(f'Structural MRI rows: {len(structural):,}\n')

# ── TEST 1: Overall paragraph count comparison ────────────────────────────────
print('=' * 60)
print('TEST 1: Paragraph count — old vs new splitter')
print('=' * 60)
structural['old_count'] = structural['findings'].apply(lambda t: len(split_report_old(t)))
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))

for label, col in [('Old splitter', 'old_count'), ('New splitter', 'new_count')]:
    s = structural[col]
    print(f'\n{label}:')
    print(f'  Mean:   {s.mean():.2f}')
    print(f'  Median: {s.median():.0f}')
    print(f'  Max:    {s.max()}')
    print(f'  >20 paragraphs (over-split): {(s > 20).sum():,} reports ({100*(s>20).mean():.1f}%)')

improved = (structural['new_count'] < structural['old_count']).sum()
regressed = (structural['new_count'] > structural['old_count']).sum()
print(f'\nImproved (fewer paragraphs):  {improved:,} reports ({100*improved/len(structural):.1f}%)')
print(f'Regressed (more paragraphs):  {regressed:,} reports ({100*regressed/len(structural):.1f}%)')

# ── TEST 2: Multi-section report handling ─────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 2: Multi-section report handling (BRAIN MRI + HEAD MRA etc.)')
print('=' * 60)
multi_section = structural[SECTION_SPLIT_PATTERN.search(structural['findings'].fillna('').astype(str)).values
    if False else structural['findings'].str.contains(
        r'HEAD MRA|NECK MRA|MRI BRAIN AND NECK', case=False, regex=True, na=False
    )]
print(f'Multi-section reports detected: {len(multi_section):,}')
if len(multi_section) > 0:
    print(f'  Old avg paragraphs: {multi_section["old_count"].mean():.1f}')
    print(f'  New avg paragraphs: {multi_section["new_count"].mean():.1f}')
    print('\nSample multi-section report — new split output:')
    sample = multi_section.iloc[0]
    paras = split_report(sample['findings'])
    for i, p in enumerate(paras[:5]):
        print(f'  [{i+1}] {p[:120]}')
    if len(paras) > 5:
        print(f'  ... ({len(paras)-5} more paragraphs)')

# ── TEST 3: Noise filter performance ─────────────────────────────────────────
print('\n' + '=' * 60)
print('TEST 3: Noise filter performance')
print('=' * 60)
structural['paragraph_list'] = structural['findings'].apply(split_report)
exploded = (
    structural.explode('paragraph_list')
    .rename(columns={'paragraph_list': 'paragraph'})
    .dropna(subset=['paragraph'])
    .reset_index(drop=True)
)
noise_mask = exploded['paragraph'].apply(is_noise)
filtered = exploded[~noise_mask].copy().reset_index(drop=True)
filtered['is_negative'] = filtered['paragraph'].apply(is_negative)
filtered['is_extracranial'] = filtered['paragraph'].apply(is_extracranial)
positive = (~filtered['is_negative'] & ~filtered['is_extracranial']).sum()

print(f'Total paragraphs after split:    {len(exploded):,}')
print(f'Dropped (noise):                 {noise_mask.sum():,} ({100*noise_mask.mean():.1f}%)')
print(f'Remaining:                       {len(filtered):,}')
print(f'  - Negative/normal (tagged):    {filtered["is_negative"].sum():,}')
print(f'  - Extracranial (tagged):       {filtered["is_extracranial"].sum():,}')
print(f'  - Positive brain findings:     {positive:,}')

# ── TEST 4: Sample output across complexity tiers ─────────────────────────────
print('\n' + '=' * 60)
print('TEST 4: Sample output — short / medium / long reports')
print('=' * 60)
structural['new_count'] = structural['findings'].apply(lambda t: len(split_report(t)))
for label, mask in [
    ('SHORT (1-2 paragraphs)',    structural['new_count'].between(1, 2)),
    ('MEDIUM (6-8 paragraphs)',   structural['new_count'].between(6, 8)),
    ('LONG (15+ paragraphs)',     structural['new_count'] >= 15),
]:
    subset = structural[mask]
    if len(subset) == 0:
        continue
    row = subset.sample(1, random_state=42).iloc[0]
    paras = split_report(row['findings'])
    print(f'\n--- {label} | {len(paras)} paragraphs ---')
    for i, p in enumerate(paras[:4]):
        neg = '[NEG] ' if is_negative(p) else '[POS] '
        ext = '[EXT] ' if is_extracranial(p) else ''
        print(f'  {neg}{ext}{p[:130]}')
    if len(paras) > 4:
        print(f'  ... ({len(paras)-4} more)')

# ── TEST 5: Spot check — 20 random positive findings ─────────────────────────
print('\n' + '=' * 60)
print('TEST 5: 20 random positive brain findings (manual review)')
print('=' * 60)
pos_sample = filtered[~filtered['is_negative'] & ~filtered['is_extracranial']].sample(20, random_state=99)
for i, (_, row) in enumerate(pos_sample.iterrows()):
    print(f'\n[{i+1}] {row["paragraph"][:200]}')

print('\n' + '=' * 60)
print('ALL TESTS COMPLETE')
print('=' * 60)

Loading data...
Structural MRI rows: 36,924

TEST 1: Paragraph count — old vs new splitter

Old splitter:
  Mean:   9.05
  Median: 8
  Max:    88
  >20 paragraphs (over-split): 1,475 reports (4.0%)

New splitter:
  Mean:   7.94
  Median: 7
  Max:    70
  >20 paragraphs (over-split): 483 reports (1.3%)

Improved (fewer paragraphs):  11,216 reports (30.4%)
Regressed (more paragraphs):  0 reports (0.0%)

TEST 2: Multi-section report handling (BRAIN MRI + HEAD MRA etc.)
Multi-section reports detected: 1,725
  Old avg paragraphs: 19.6
  New avg paragraphs: 7.0

Sample multi-section report — new split output:
  [1] Brain Parenchyma: There are extensive regions of T2 prolongation in the white matter likely manifestation of chronic sma
  [2] Ventricular System and Extra-Axial Spaces: The ventricles and cortical sulci are prominent. Cavum septum pellucidum is i
  [3] Skull Base, [REDACTED], and Visualized Upper Cervical Spine: Normal.
  [4] Paranasal Sinuses, Mastoids, and Orbits: Mucosal thick